# 第13回　多変量解析への招待：主成分分析（PCA）
## ―― たくさんの変数の背後にある、少数の軸を見つける

統計学Ⅰ（B）　／　北星学園大学

注目は ――

> 変数がたくさんあっても、**本質はもっと少ない次元**で表せることが多い。

### フック

> 霊長類データには、体重・妊娠期間・離乳日齢・寿命・産子数… とたくさんの変数がある。
> **全部の関係を一度に見るのは無理**（変数5個でも散布図は10通り、10個なら45通り）。
> 
> これらの背後にある「**本当の軸**」は、いくつあるのだろう？

第5回で「生き物のデータは何もかも相関する」と見た。**相関しているということは、共通の何かが動かしている**ということでもある。今日はそれを取り出す。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    # 原典（PanTHERIA）から組み立て直す。まずリポジトリ同梱の複製、だめなら発行元から。
    # 発行元は User-Agent を見て弾くことがあるため、明示して取得する。
    import io, urllib.request
    SRCS = [
        "https://raw.githubusercontent.com/aonoa68/toukei-1/main/docs/data/PanTHERIA_1-0_WR05_Aug2008.txt.gz",
        "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt",
    ]
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = None
    for _url in SRCS:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=60) as _r:
                _raw = _r.read()
            _comp = "gzip" if _url.endswith(".gz") else None
            src = pd.read_csv(io.BytesIO(_raw), sep="\t", compression=_comp)
            break
        except Exception:
            continue
    if src is None:
        raise RuntimeError("原典データを取得できませんでした")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 1. 主成分分析（PCA）の発想

**生活史**に関する5つの変数を使う：**体重・妊娠期間・離乳日齢・最長寿命・一腹産子数**。

PCAは、これら5次元のデータを、**ばらつき（情報）が最も大きい方向**に新しい軸を引き直す。最初の軸（第1主成分）が一番多くの情報を持ち、次の軸（第2主成分）がその次…と続く。少数の軸だけ残せば**次元を削減**できる。

> **前処理を2つ行う。**
> ① 何桁にもまたがる量なので**対数**にする（第3回以来のいつもの処理）
> ② 変数ごとに単位が違う（g・日・月・頭）ので**標準化**して土俵をそろえる

In [ ]:
cols = ["体重g", "妊娠期間日", "離乳日齢", "最長寿命月", "一腹産子数"]

s = df.dropna(subset=cols).copy()
for c in cols:
    s = s[s[c] > 0]
for c in cols:
    s[c] = np.log10(s[c])          # ① 対数

X = StandardScaler().fit_transform(s[cols])   # ② 標準化
pca = PCA().fit(X)

print(f"5つの変数すべてに記録がある種 = {len(s)} 種\n")
print("寄与率（各主成分が説明する情報の割合）")
cum = 0
for i, r in enumerate(pca.explained_variance_ratio_):
    cum += r
    print(f"  第{i+1}主成分: {r*100:5.1f}%   （累積 {cum*100:5.1f}%）")

**第1主成分だけで全体の約74%**、第2主成分まで合わせると約87%の情報を説明できる。5次元のデータの本質は、ほぼ**2次元**、いや**ほとんど1次元**に圧縮できるということだ。

では、その第1主成分は「何の軸」なのか？　各変数がどれくらい効いているか（**負荷**）を見て、人間が**命名**する。

In [ ]:
負荷 = pd.DataFrame(pca.components_[:2].T, index=cols, columns=["第1主成分", "第2主成分"])
print(負荷.round(2))

### 第1主成分を読む

| 変数 | 負荷 | 向き |
|---|---:|---|
| 体重 | +0.48 | 大きいほど＋ |
| 妊娠期間 | +0.45 | 長いほど＋ |
| 離乳日齢 | +0.48 | 遅いほど＋ |
| 最長寿命 | +0.42 | 長いほど＋ |
| **一腹産子数** | **−0.39** | **少ないほど＋** |

4つが同じ向きで、**産子数だけが逆向き**。これを言葉にすると ――

> **＋の側**：体が大きく、妊娠が長く、乳離れが遅く、長生きし、**一度に少ししか産まない**
> **−の側**：体が小さく、妊娠が短く、乳離れが早く、短命で、**一度にたくさん産む**

これは生物学で **生活史の「速い－遅い」連続体（fast–slow continuum）** と呼ばれるものである。

- **速い戦略**：短く生きて、たくさん産む（数で勝負）
- **遅い戦略**：長く生きて、少なく産んで、大事に育てる（質で勝負）

**5つの変数の背後に、1本の生き方の軸が隠れていた。**これが**次元削減＝本質の抽出**だ。

---
## 2. 2次元に圧縮して可視化

各種を、第1主成分（速い－遅い）と第2主成分の2軸で配置する。**科ごとに色を変える。**

**注意：PCAには「科」の情報を一切与えていない。** 5つの数値だけから軸を作った。

In [ ]:
Z = PCA(n_components=2).fit_transform(X)
s = s.assign(PC1=Z[:, 0], PC2=Z[:, 1])

主要科 = s["科"].value_counts()
主要科 = 主要科[主要科 >= 4].index

plt.figure(figsize=(8, 5.5))
for fam in 主要科:
    g = s[s["科"] == fam]
    plt.scatter(g["PC1"], g["PC2"], s=40, alpha=0.8, label=f"{fam} (n={len(g)})")
その他 = s[~s["科"].isin(主要科)]
plt.scatter(その他["PC1"], その他["PC2"], s=25, alpha=0.4, color="gray", label="その他")

plt.xlabel("第1主成分　←速い戦略　　　遅い戦略→")
plt.ylabel("第2主成分")
plt.title("5次元を2次元に圧縮（情報の約87%を保持）")
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.legend(fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()

**科がきれいに並んだ。** 数値だけから作った軸なのに、分類群が左右に整列している。

In [ ]:
順位 = (s.groupby("科")["PC1"]
          .agg(種数="count", PC1平均="mean")
          .round(2).sort_values("PC1平均"))
print("科別の第1主成分の平均（小さいほど『速い』戦略）")
print(順位.to_string())

**コビトキツネザル科（−4.31）から、ヒト科（+3.50）まで、一直線に並んでいる。**

そして端を見てほしい。

In [ ]:
print("最も『速い』5種:")
print(s.nsmallest(5, "PC1")[["学名", "科", "PC1"]].to_string(index=False))
print("\n最も『遅い』5種:")
print(s.nlargest(5, "PC1")[["学名", "科", "PC1"]].to_string(index=False))

### 遅い側の端に、私たちがいる

**最も「遅い」5種はすべてヒト科**で、その頂点が **`Homo sapiens`（+4.15）**である。

霊長類のなかで、ヒトは **最も体が大きい部類で、妊娠が長く、乳離れが遅く、最も長生きし、一度にほとんど1人しか産まない** ―― 生活史戦略の極北にいる。

反対の端はネズミキツネザル（`Microcebus rufus`、−5.55）。体重50g足らず、短命で、複数の子を産む。

> **PCAは「ヒトは特別だ」と教えたわけではない。**
> 5つの数値を、ばらつきが最大になる向きに並べ直しただけである。> 結果として分類群が並び、ヒトが端に来た。**それをどう意味づけるかは、人間の仕事**である。

---
## 3. 大事な注意

- **PCAは情報を失う**。今回、第1・第2主成分で説明できたのは約**87%**。残りの**約13%は捨てている**。圧縮はタダではない。
- **主成分に必ず意味があるとは限らない**。第1主成分は「速い－遅い」と読めたが、**第2主成分（13.4%）は簡単には命名できない**（産子数と寿命が同じ向きに乗っており、生活史の軸では説明しにくい）。現実のデータでは「何だかよく分からない軸」のほうが多い。**意味づけ（命名）は人間の解釈**であり、数学が保証してくれるものではない。
- 主成分の**向き（符号）は反転することがある**（右が「遅い」でも左が「遅い」でも数学的には同じ。実行環境によって入れ替わりうる）。
- **使った89種は「5変数すべてに記録がある種」だけ**である（全376種の24%）。第6回で見たとおり、よく研究された種に偏っている。**この軸は、よく調べられた霊長類の軸**である。

> ❌ よくある誤り：「PCAで情報は失われない」「主成分には必ずはっきりした意味がある」「PCAが分類群を見つけた＝分類の正しさを証明した」。

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 次元削減 | たくさんの変数を、少数の軸（主成分）にまとめる |
| 主成分 | ばらつき（情報）が最大の方向に引いた新しい軸 |
| 寄与率 | 各主成分が説明する情報の割合（第1主成分74%・累積87%） |
| 負荷と命名 | 各変数の効き方を見て、人間が軸に意味を与える（例：速い－遅い） |
| 前処理 | 桁が違うなら対数、単位が違うなら標準化 |
| ❌ 誤り | PCAで情報は失われない／主成分には必ず明確な意味がある |

> **PCAは『たくさんの変数の背後にある少数の軸』を見つける道具。**
> ただし情報は一部失われ、軸の意味づけは人間の解釈。

> **第5回とのつながり。**「生き物のデータは何もかも相関する」のは、> **背後に共通の軸があるから**だった。第5回では交絡＝厄介者として扱ったが、> PCAはそれを**取り出すべき構造**として扱う。**同じ現象を、目的によって違う名前で呼んでいる。**

**課題（Moodle）**：PCAの寄与率と負荷を読み、第1・第2主成分が「何を表す軸か」を命名する。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。